# 00 · Preparación `paper95` en Kaggle (CPU)
Usa el dataset Kaggle `thngdngvn/openneuro-ds004504` (derivados AD/CN de OpenNeuro ds004504), ejecuta QC y crea un bundle verificable. La lógica científica vive en `src/` y `scripts/`.

In [ ]:
from pathlib import Path
import os, shutil
assert Path('/kaggle/working').exists(), 'Este notebook debe ejecutarse en Kaggle.'
assert os.environ.get('KAGGLE_KERNEL_RUN_TYPE'), 'No se detectó el entorno de ejecución Kaggle.'
candidates = [p.parent for p in Path('/kaggle/input').glob('**/pyproject.toml') if (p.parent / 'src/fpsd_reassessment').exists()]
if Path('/kaggle/working/fpsd-cnn-reassessment/pyproject.toml').exists():
    REPO = Path('/kaggle/working/fpsd-cnn-reassessment')
elif len(candidates) == 1:
    REPO = Path('/kaggle/working/fpsd-cnn-reassessment')
    shutil.copytree(candidates[0], REPO, dirs_exist_ok=True)
else:
    raise RuntimeError('Añada una copia privada del repositorio como Kaggle Input; debe contener pyproject.toml.')
print('Repositorio:', REPO)

In [ ]:
import sys, importlib, json
sys.path.insert(0, str(REPO/'src'))
os.environ['PYTHONPATH'] = str(REPO/'src')
versions = {}
for name in ('numpy', 'scipy', 'pandas', 'sklearn', 'mne', 'pyarrow', 'yaml', 'psutil', 'matplotlib', 'seaborn', 'tqdm', 'pytest'):
    module = importlib.import_module(name)
    versions[name] = getattr(module, '__version__', 'unknown')
print(json.dumps(versions, indent=2))

In [ ]:
import json, platform, psutil, datetime
print(json.dumps({'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': platform.python_version(), 'cpu': platform.processor(), 'logical_cpu': psutil.cpu_count(), 'ram_gib': round(psutil.virtual_memory().total/2**30, 2), 'disk_free_gib': round(psutil.disk_usage('/kaggle/working').free/2**30, 2)}, indent=2))

In [ ]:
import subprocess
DATASET_DIR = Path('/kaggle/input/datasets/thngdngvn/openneuro-ds004504/ds004504')
assert (DATASET_DIR/'participants.tsv').exists() and (DATASET_DIR/'derivatives').exists()
subprocess.run([sys.executable, str(REPO/'scripts/prepare_features.py'), '--dataset-dir', str(DATASET_DIR), '--output-dir', '/kaggle/working/fpsd_features', '--config', str(REPO/'configs/features/paper95.yaml'), '--skip-download'], cwd=REPO, check=True)

In [ ]:
import numpy as np, pandas as pd
bundle = Path('/kaggle/working/fpsd_features')
with np.load(bundle/'paper95_features.npz') as z:
    print('X:', z['X'].shape, 'X_flat:', z['X_flat'].shape)
    assert z['X'].shape[1:] == (19, 5) and z['X_flat'].shape[1] == 95
    assert np.isfinite(z['X']).all()
meta = pd.read_parquet(bundle/'windows_metadata.parquet')
qc = pd.read_csv(bundle/'subjects_qc.csv')
assert meta.subject_id.nunique() == 65 and len(qc) == 65
assert Path('/kaggle/working/fpsd_features_bundle.zip').exists()
print('Bundle listo:', '/kaggle/working/fpsd_features_bundle.zip')